In [1]:
#234567890#234567890#234567890#234567890#234567890#234567890#234567890#23456789
import os

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else 'cpu')
device

'mps'

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10))

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
mod = NeuralNetwork().to(device)
mod

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)

In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = mod(X)
pred_prob = nn.Softmax(dim=1)(logits)
y_pred = pred_prob.argmax(1)
y_pred

tensor([7], device='mps:0')

In [6]:
input_image = torch.rand(3, 28, 28)
input_image.size()

torch.Size([3, 28, 28])

In [7]:
flatten = nn.Flatten()
flat_img = flatten(input_image)
flat_img.size()

torch.Size([3, 784])

In [8]:
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
hidden1 = layer1(flat_img)
hidden1.size()

torch.Size([3, 20])

In [9]:
print(hidden1)
hidden1 = nn.ReLU()(hidden1)
print()
print(hidden1)

tensor([[-3.1728e-01, -2.2679e-01, -8.2071e-01, -1.5656e-01,  3.0691e-01,
         -2.9199e-01,  1.7191e-01,  2.5479e-01, -7.2703e-01,  2.2842e-01,
         -1.6229e-01,  5.0918e-01,  3.5948e-01,  1.6020e-01,  2.3713e-01,
         -4.0697e-01, -6.0432e-01,  1.1135e-01,  3.1171e-01, -2.3175e-01],
        [-3.1253e-01, -2.0789e-01, -6.7568e-01,  9.4041e-03,  3.0427e-01,
         -5.0646e-01,  1.8344e-01,  1.1484e-01, -4.8499e-01,  6.3155e-01,
         -1.1345e-01,  1.4614e-02,  2.3681e-01,  2.0471e-01,  9.0826e-02,
         -2.4996e-01, -3.6271e-01, -1.8499e-01,  4.7949e-01, -4.1944e-01],
        [-6.0219e-03, -1.9115e-01, -4.1543e-01,  5.1258e-02,  2.3738e-01,
         -4.4187e-01,  2.2527e-01,  4.3921e-01, -9.1092e-01,  4.2126e-01,
         -1.1241e-04,  2.7956e-01,  2.3145e-01, -3.6453e-01,  3.2051e-03,
         -2.2489e-01, -1.6714e-01, -7.4354e-02,  5.6208e-01, -3.1549e-01]],
       grad_fn=<AddmmBackward0>)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.3069, 0.0000, 0.1719, 0.2548, 0

In [10]:
seq_mod = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10))
input_img = torch.rand(3, 28, 28)
logits = seq_mod(input_image)
logits

tensor([[ 0.1083, -0.3304, -0.1867, -0.0825, -0.1046,  0.2491, -0.0341,  0.1871,
         -0.0940,  0.1727],
        [ 0.2791, -0.2749, -0.1874, -0.1156, -0.0949,  0.2352,  0.0197,  0.0173,
          0.0483, -0.0431],
        [ 0.1081, -0.3974, -0.1386, -0.1200, -0.0379,  0.1804,  0.0050,  0.0728,
         -0.0403,  0.1090]], grad_fn=<AddmmBackward0>)

In [11]:
softmax = nn.Softmax(dim=1)
probs = softmax(logits)
probs

tensor([[0.1110, 0.0716, 0.0827, 0.0917, 0.0897, 0.1278, 0.0963, 0.1201, 0.0907,
         0.1184],
        [0.1319, 0.0758, 0.0827, 0.0889, 0.0908, 0.1263, 0.1018, 0.1015, 0.1047,
         0.0956],
        [0.1130, 0.0682, 0.0883, 0.0899, 0.0976, 0.1215, 0.1019, 0.1091, 0.0974,
         0.1131]], grad_fn=<SoftmaxBackward0>)

In [13]:
print(f'Model structure: {mod}\n\n')
for name, param in mod.named_parameters():
    print(f'Layer: {name} | Size: {param.size()} | Values: {param[:2]}\n')

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0055, -0.0101, -0.0107,  ..., -0.0299, -0.0083, -0.0207],
        [ 0.0323, -0.0154, -0.0125,  ...,  0.0293,  0.0146, -0.0297]],
       device='mps:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([ 0.0260, -0.0232], device='mps:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[-0.0250, -0.0173, -0.0202,  ...,  0.0170, -0.0257, -0.0318],
        [-0.0395,  0.0219,  0.0341,  ...,  0.0305, -0.0224, -0.0093]],
       device='mps:0', grad_fn=<SliceBackw